# Time Shifting and Rolling Windows

When analyzing sequential data (like sales over time or stock prices), we often need to look across rows. For example, comparing today's revenue to yesterday's revenue, or calculating a 3-day moving average to smooth out spikes.

We do this using two highly optimized Pandas techniques:
*   **`.shift(n)`**: Moves your data column down (positive `n`) or up (negative `n`) by a specified number of rows. This is ideal for row-to-row comparisons.
*   **`.rolling(window=n)`**: Creates a rolling window of `n` periods to compute moving statistics (like rolling sums, averages, or standard deviations).

### Real-World Analogy
*   **`.shift(1)`**: Looking at a calendar and writing down *yesterday's* weather on today's page so you can easily compare them side-by-side.
*   **`.rolling(window=7).mean()`**: Finding a student's average test performance over their last 7 quizzes, updating the average every time they take a new test to see their current progress trend.

### Code Examples

Let's create a DataFrame tracking daily sales revenue:


In [1]:
import pandas as pd

sales = pd.DataFrame({
    'Day': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
    'Revenue': [100, 120, 90, 110, 150, 200, 180]
})
print("--- Daily Sales ---")
print(sales)

--- Daily Sales ---
   Day  Revenue
0  Mon      100
1  Tue      120
2  Wed       90
3  Thu      110
4  Fri      150
5  Sat      200
6  Sun      180



#### Time Shifting (`.shift()`)
Let's shift the revenue column to compare each day's performance to the previous day:

In [2]:
# Shift the Revenue column down by 1 row
sales['Prev_Day_Revenue'] = sales['Revenue'].shift(1)

# Calculate daily growth
sales['Daily_Revenue_Growth'] = sales['Revenue'] - sales['Prev_Day_Revenue']
print(sales)

   Day  Revenue  Prev_Day_Revenue  Daily_Revenue_Growth
0  Mon      100               NaN                   NaN
1  Tue      120             100.0                  20.0
2  Wed       90             120.0                 -30.0
3  Thu      110              90.0                  20.0
4  Fri      150             110.0                  40.0
5  Sat      200             150.0                  50.0
6  Sun      180             200.0                 -20.0


Note: The first row gets a `NaN` (not-a-number) value because there is no historical row preceding Monday.

#### B) Rolling Window Calculations (`.rolling()`)
Let's calculate a **3-day rolling sum** and **3-day rolling average** of the revenue :



In [3]:
# Calculate a 3-day rolling sum
sales['3Day_Rolling_Sum'] = sales['Revenue'].rolling(window=3).sum()

# Calculate a 3-day rolling mean
sales['3Day_Rolling_Mean'] = sales['Revenue'].rolling(window=3).mean()
print(sales)

   Day  Revenue  Prev_Day_Revenue  Daily_Revenue_Growth  3Day_Rolling_Sum  \
0  Mon      100               NaN                   NaN               NaN   
1  Tue      120             100.0                  20.0               NaN   
2  Wed       90             120.0                 -30.0             310.0   
3  Thu      110              90.0                  20.0             320.0   
4  Fri      150             110.0                  40.0             350.0   
5  Sat      200             150.0                  50.0             460.0   
6  Sun      180             200.0                 -20.0             530.0   

   3Day_Rolling_Mean  
0                NaN  
1                NaN  
2         103.333333  
3         106.666667  
4         116.666667  
5         153.333333  
6         176.666667  


*Why are the first two rows `NaN`?*
By default, the rolling window requires a complete set of rows (in this case, 3 rows) to perform the math. If you want to allow calculations with fewer than 3 values at the start, you can set the `min_periods=1` parameter:
`sales['Revenue'].rolling(window=3, min_periods=1).mean()`

### Common Pitfalls to Avoid
1.  **Forgetting `min_periods`**: If you do not specify `min_periods`, the starting elements of your rolling column will always be filled with `NaN`.
2.  **Shifting with Grouped Data**: If you run `.shift()` on an un-grouped dataset containing multiple different stores or categories, rows from one category will shift into another. Always group your shifts for multi-category datasets: `df.groupby('Category')['Sales'].shift(1)`.


#### Exercise 1 (Hard)
You have a sequential stock price tracker:
```python
import pandas as pd
stock = pd.DataFrame({
    'Company': ['Apple', 'Apple', 'Apple', 'Google', 'Google', 'Google'],
    'Day': [1, 2, 3, 1, 2, 3],
    'Price': [150, 155, 152, 2800, 2820, 2810]
})
```
Write Pandas code to calculate a **2-day rolling average price** independently for each company. (Hint: combine `.groupby()` with `.rolling()`).


In [4]:
import pandas as pd

stock = pd.DataFrame({
    'Company': ['Apple', 'Apple', 'Apple', 'Google', 'Google', 'Google'],
    'Day': [1, 2, 3, 1, 2, 3],
    'Price': [150, 155, 152, 2800, 2820, 2810]
})

# Calculate 2-day rolling mean per company
# We must reset the index afterward because grouping + rolling creates a MultiIndex
stock['2Day_Rolling_Avg'] = stock.groupby('Company')['Price'].rolling(window=2, min_periods=1).mean().reset_index(level=0, drop=True)
print(stock)

  Company  Day  Price  2Day_Rolling_Avg
0   Apple    1    150             150.0
1   Apple    2    155             152.5
2   Apple    3    152             153.5
3  Google    1   2800            2800.0
4  Google    2   2820            2810.0
5  Google    3   2810            2815.0
